In [5]:
import arcpy
import pandas as pd

In [6]:
# Define paths
gdb_path = r"C:\Demo\PostgreSQL-192-sewerassetsdb(NWdataowner).sde"
excel_path = r"C:\Demo\nwsc_gdb_domains.xlsx"
sheet_name = "Domains"  # Name of the tab in Excel

# Read the Excel sheet using pandas
df = pd.read_excel(excel_path, sheet_name=sheet_name)

# Get a list of existing domains in the GDB to avoid duplicate errors
existing_domains = [d.name for d in arcpy.da.ListDomains(gdb_path)]

# Group the data by Domain Name to process one domain at a time
grouped = df.groupby('DomainName')

for domain_name, group in grouped:
    # Extract domain metadata from the first row of the group
    first_row = group.iloc[0]
    domain_type = str(first_row['DomainType']).upper().strip()  # CODED or RANGE
    field_type = str(first_row['FieldType']).upper().strip()    # TEXT, SHORT, LONG, etc.
    
    print(f"Processing domain: {domain_name} ({domain_type})...")
    
    # 1. Create the Domain container if it doesn't exist
    if domain_name not in existing_domains:
        arcpy.management.CreateDomain(
            in_workspace=gdb_path,
            domain_name=domain_name,
            domain_description=f"Imported {domain_name} domain",
            field_type=field_type,
            domain_type=domain_type
        )
        print(f" -> Created empty domain: {domain_name}")
    else:
        print(f" -> Domain {domain_name} already exists. Adding/updating values.")

    # 2. Populate the Domain based on its type
    if domain_type == "CODED":
        for index, row in group.iterrows():
            code = row['CodeOrMin']
            description = row['DescriptionOrMax']
            
            # Add coded value (wrapped in try/except in case code already exists)
            try:
                arcpy.management.AddCodedValueToDomain(
                    in_workspace=gdb_path,
                    domain_name=domain_name,
                    code=code,
                    code_description=description
                )
            except arcpy.ExecuteError as e:
                # Catches error if the specific code already exists in that domain
                print(f"    ! Code '{code}' could not be added (may already exist).")
                
    elif domain_type == "RANGE":
        # For a range domain, we just need the min and max from the first row
        min_val = first_row['CodeOrMin']
        max_val = first_row['DescriptionOrMax']
        
        arcpy.management.SetValueForRangeDomain(
            in_workspace=gdb_path,
            domain_name=domain_name,
            min_value=min_val,
            max_value=max_val
        )
        print(f" -> Set range boundary to: {min_val} - {max_val}")

print("\nAll domains processed successfully!")

Processing domain: 5_Asset_Type_Water_Device_Fire_Hydrant (CODED)...
 -> Created empty domain: 5_Asset_Type_Water_Device_Fire_Hydrant
Processing domain: 5_Asset_Type_Water_Device_Pressure_Valve (CODED)...
 -> Created empty domain: 5_Asset_Type_Water_Device_Pressure_Valve
Processing domain: 5_Asset_Type_Water_Device_Storage (CODED)...
 -> Created empty domain: 5_Asset_Type_Water_Device_Storage
Processing domain: 5_Asset_Type_Water_Device_System_Meter (CODED)...
 -> Created empty domain: 5_Asset_Type_Water_Device_System_Meter
Processing domain: FixtureData_fixturetype_686b2755-5e68-4b18-8b7c-6119cbc84d89_1 (CODED)...
 -> Created empty domain: FixtureData_fixturetype_686b2755-5e68-4b18-8b7c-6119cbc84d89_1
Processing domain: FixtureData_territory_0b0ec27d-7fc1-4205-871b-c423b86e8ab1_1 (CODED)...
 -> Created empty domain: FixtureData_territory_0b0ec27d-7fc1-4205-871b-c423b86e8ab1_1
Processing domain: Map_FixtureData_fieldstaff_2498938f6fd9448a84165f48cf725ec8_1_1_1_1_1_1_1_1_1_1_1_1_1_1_1_1